In [21]:
import os

import pandas as pd
import numpy as np

import regex as re
from datetime import datetime

In [22]:
data_path = "../data/in/"
output_path = "../data/out/"

regions = {"vor": "20241214-0617_gtfs_vor_2024", #vienna, lower austria, burgenland
           "ooevv": "20241212-0156_gtfs_ooevv_2024", #upper austria
           "esg": "20241203-0058_gtfs_esg_2024", #linz
           "verbundlinie": "20241217-0310_gtfs_verbundlinie_2024", #styria
           "kaernterlinien": "20241214-0253_gtfs_kaerntnerlinien_2024", #carinthia
           "salzburgverkehr": "20241217-0359_gtfs_salzburgverkehr_2024", #salzburg
           "vvt": "20241217-0436_gtfs_vvt_2024", #tyrol
           "vmobil": "20241212-0624_gtfs_vmobil_2024", #vorarlberg
           "obb": "GTFS_2024_obb"} #oebb maybe 20241217-0222_gtfs_evu_2024

# select region
state_name = "obb"
# select day for calculation in format YYYYMMDD in 2024
selected_day = 20240530

# stop categories
table_roman = np.array([
    ["I", "I", "II", "III"],        # < 5 min
    ["I", "II", "III", "III"],      # 5 >= x <= 10
    ["II", "III", "IV", "IV"],      # 10 < x < 20
    ["III", "IV", "V", "V"],        # 20 >= x < 40
    ["IV", "V", "VI", "VI"],        # 40 >= x <= 60
    ["V", "VI", "VII", "VII"],      # 60 < x <= 120  
    ["X", "VII", "VIII", "VIII"],    # 120 < x <= 210 
    ["X", "X", "X", "X"],               # > 210 
                                    # X = empty, i.e. worst case
])
table = np.array([
    [0, 0, 0, 0],        # < 5 min
    [0, 1, 2, 2],        # 5 >= x <= 10
    [1, 2, 3, 3],        # 10 < x < 20
    [2, 3, 4, 4],        # 20 >= x < 40
    [3, 4, 5, 5],        # 40 >= x <= 60
    [4, 5, 6, 6],        # 60 < x <= 120  
    [-1, 6, 7, 7],       # 120 < x <= 210 
    [-1, -1, -1, -1],    # > 210
])

# transport_category = ["Fernverkehr REX", 
#                       "S-Bahn / U-Bahn, Regionalbahn, Schnellbus, Lokalbahn", 
#                       "Straßenbahn, Metrobus, 0-Bus", 
#                       "Bus"]
route_type_translation = {0: 2, 1: 1, 2: 0, 3: 3, 11: 3,} #7: 3, 4: 3



In [23]:
def lookup_category(interval, t_cat):
    """
    Lookup the stop category based on the interval and the transport type.
    """
    if interval < 5:
        return table[0][t_cat]
    elif interval <= 10:
        return table[1][t_cat]
    elif interval < 20:
        return table[2][t_cat]
    elif interval < 40:
        return table[3][t_cat]
    elif interval <= 60:
        return table[4][t_cat]
    elif interval <= 120:
        return table[5][t_cat]
    elif interval <= 210:
        return table[6][t_cat]
    else:
        return table[7][t_cat]
    
def category_to_roman(t_cat, reverse=False):
    """
    Convert a category to a roman numeral or vice versa if reverse is True
    """
    roman_numerals = ["I", "II", "III", "IV", "V", "VI", "VII", "VIII", "X"]
    if reverse:
        return roman_numerals.index(t_cat)
    return roman_numerals[t_cat]
    
def detect_route_type(trip_name, route_type):
    """
    Detect the type of transportation based on the trip name and the route type
    By default, the transport type is only based on the route type.
    For route type 2 (train), the trip_name is used, to further distinguish between different trains, if available.
    """
    # TODO: also consider route_short/long_name ??? 
    if route_type == 2 and not pd.isna(trip_name):
        trips_name = trip_name.lower()
        if any(x in trips_name for x in ["rj", "rjx", "nj", "en", "ic", "ec", "ice", "ecb", "rex", "wb", "rgj", "cjx" ]): #fernverkehr
            return 0
        else:
            return 1
    else:
        return route_type_translation[route_type]

In [24]:
# read in the data
path = f"{data_path}/{regions[state_name]}/"

stops = pd.read_csv(path + "/stops.txt", quotechar='"', sep=",")
stop_times = pd.read_csv(path + "/stop_times.txt", quotechar='"', sep=",")
trips = pd.read_csv(path + "/trips.txt", quotechar='"', sep=",")
routes = pd.read_csv(path + "/routes.txt", quotechar='"', sep=",")
calendar = pd.read_csv(path + "/calendar.txt", quotechar='"', sep=",")
calendar_dates = pd.read_csv(path + "/calendar_dates.txt", quotechar='"', sep=",")

print(stops.shape)
stops.head()

(6631, 9)


/var/folders/t9/cqlkdlt50h94khlrvmtm0zbc0000gn/T/ipykernel_39306/916577950.py:5: DtypeWarning: Columns (5,6,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  stop_times = pd.read_csv(path + "/stop_times.txt", quotechar='"', sep=",")


,stop_id,stop_name,stop_lat,stop_lon,zone_id,location_type,parent_station,level_id,platform_code
0,at:41:3087:0:1,Bad Sauerbrunn Bahnhof,47.773870,16.324886,NaN,NaN,NaN,Level 0,1.0
1,at:41:3087:0:2,Bad Sauerbrunn Bahnhof,47.773858,16.324859,NaN,NaN,NaN,Level 0,2.0
2,at:41:3087:2,Bus,47.774226,16.324509,NaN,NaN,Pat:41:3087,Level 0,NaN
3,at:41:3087:3,Zugang,47.773949,16.324356,NaN,2.0,Pat:41:3087,Level 0,NaN
4,at:41:3087:4,P+R,47.772578,16.325695,NaN,3.0,Pat:41:3087,Level 0,NaN


In [25]:
# find weekday of selected day
# days = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
# date = datetime.fromisoformat(str(selected_day))
# day_string = days[date.weekday()]

date = datetime.fromisoformat(str(selected_day))
day_string = date.strftime("%A").lower()

calendar_filtered = calendar[(calendar['start_date'] <= selected_day) & (calendar['end_date'] >= selected_day) & (calendar[day_string] == 1)].copy()
#calendar_dates_filtered = calendar_dates[calendar_dates["date"] == selected_day]

added_service = calendar_dates[(calendar_dates['date'] == selected_day) & (calendar_dates['exception_type'] == 1)]
removed_service = calendar_dates[(calendar_dates['date'] == selected_day) & (calendar_dates['exception_type'] == 2)]

# only keep services that run on that weekday
# calendar_filtered = calendar_filtered[calendar_filtered[day_string] == 1]

print(calendar_filtered.shape)
added_service.head()

(1232, 10)


,service_id,date,exception_type
259,SaSo,20240530,1
988,TA+03400,20240530,1
2706,TA+09300,20240530,1
2963,TA+0a500,20240530,1
4022,TA+0d,20240530,1


In [26]:
# keep only valid trips
trips_filtered = trips[trips['service_id'].isin(calendar_filtered['service_id'])]
print(trips_filtered.shape)

# remove services with exception_type = 2 from calendar_dates
#trips_filtered = trips_filtered[trips_filtered["service_id"].isin(calendar_dates_filtered[calendar_dates_filtered["exception_type"] == 2]["service_id"])]
trips_filtered = trips_filtered[~trips_filtered["service_id"].isin(removed_service["service_id"])]
print(trips_filtered.shape)

# add services with exception_type = 1 from calendar_dates
trips_full = pd.concat([trips_filtered, trips[trips["service_id"].isin(added_service["service_id"])]])
print(trips_full.shape)
trips_full.head()

(9328, 8)
(3492, 8)
(5307, 8)


,route_id,service_id,trip_id,shape_id,trip_headsign,trip_short_name,direction_id,block_id
3,10-A10-j24-1,TA+qf300,4.TA.10-A10-j24-1.4.R,10-A10-j24-1.4.R,Villach Hauptbahnhof,RJ 639,1.0,NaN
6,10-A11-j24-1,TA+32100,1.TA.10-A11-j24-1.1.R,10-A11-j24-1.1.R,Budapest-Keleti,RJX 165,1.0,1363.0
34,10-A11-j24-1,TA+on,35.TA.10-A11-j24-1.16.H,10-A11-j24-1.16.H,Feldkirch Bahnhof,RJX 566,0.0,NaN
46,10-A11-j24-1,TA+32100,46.TA.10-A11-j24-1.7.R,10-A11-j24-1.7.R,Flughafen Wien Bahnhof,RJX 565,1.0,NaN
62,10-A12-j24-1,TA,10.TA.10-A12-j24-1.9.R,10-A12-j24-1.9.R,Praha hlavní nádraží,RJ 372,1.0,1402.0


In [27]:
# merge trips with routes information
routes_trips = pd.merge(trips_full, routes, on='route_id', how='left')

# keep only valid trips route_type, remove e.g funicular 7 and ferry 4
routes_trips = routes_trips[routes_trips['route_type'].isin([0, 1, 2, 3, 11])]
# TODO: check if this is correct

# translate route type
routes_trips['trip_short_name'] = routes_trips['trip_short_name'].astype('str')
routes_trips['rank'] = routes_trips.apply(lambda x: detect_route_type(x['trip_short_name'], x['route_type']), axis=1)

print(routes_trips.shape)
routes_trips[routes_trips['rank'].isna()]

(5307, 13)


,route_id,service_id,trip_id,shape_id,trip_headsign,trip_short_name,direction_id,block_id,agency_id,route_short_name,route_long_name,route_type,rank


In [ ]:
# prepare stops and stop_times
stops_filtered = stops.copy()
stops_filtered['stop_id'] = stops_filtered['stop_id'].astype(str).apply(
    lambda x: (
        re.match(r'^((?:[^:]*:){3})', x).group(1).rstrip(':')
        if re.match(r'^((?:[^:]*:){3})', x)
        else x
    )
)
# keep only stop entry for parent station, if no parent station is given, keep a stop entry
stops_parents = stops_filtered[stops_filtered['stop_id'].str.startswith('Pat')].copy()
stops_parents['stop_id'] = stops_parents['stop_id'].apply(lambda x: x if x[0] != 'P' else x[1:])
stops_filtered = stops_filtered[stops_filtered['stop_id'].str.startswith('at') | stops_filtered['stop_id'].str.startswith('obb')]
stops_filtered = stops_filtered[~stops_filtered['stop_id'].isin(stops_parents['stop_id'])].drop_duplicates(subset=['stop_id'], keep='first')
# TODO: maybe filter out special stations e.g. obb_CP_80854 Wattens Sammelpunkt Bahnhofstraße MPREIS or Pat:42:99979_HoB

stops_filtered_final = pd.concat([stops_parents, stops_filtered])
print(stops_filtered_final.shape)

stop_times_filtered = stop_times.copy()
stop_times_filtered = stop_times_filtered[stop_times_filtered['departure_time'].between('06:00:00', '20:00:00')]
stop_times_filtered = stop_times_filtered[stop_times_filtered['stop_id'].str.startswith('at')]
# TODO: check if Parent station is in stop_times and keep those
stop_times_filtered['stop_id'] = stop_times_filtered['stop_id'].astype(str).apply(
    lambda x: (
        re.match(r'^((?:[^:]*:){3})', x).group(1).rstrip(':')
        if re.match(r'^((?:[^:]*:){3})', x)
        else x
    )
)

# display(stop_times_df_obb_mod.head())
#stop_times_filtered

(1194, 9)


,stop_id,stop_name,stop_lat,stop_lon,zone_id,location_type,parent_station,level_id,platform_code
6556,at:42:99979_HoB,Hst. ohne Bereich,0.000000,0.000000,NaN,3.0,Pat:42:99979,Level 0,NaN
6557,at:43:99980_HoB,Hst. ohne Bereich,0.000000,0.000000,NaN,3.0,Pat:43:99980,Level 0,NaN
6558,at:43:99986_HoB,Hst. ohne Bereich,0.000000,0.000000,NaN,3.0,Pat:43:99986,Level 0,NaN
6559,at:43:99987_HoB,Hst. ohne Bereich,0.000000,0.000000,NaN,3.0,Pat:43:99987,Level 0,NaN
6560,at:44:44349_STD,Nullbereich,48.315003,14.487643,NaN,NaN,Pat:44:44349,Level 0,NaN


In [29]:
stop_times_trips = pd.merge(stop_times_filtered, routes_trips, on='trip_id', how='inner')
print(stop_times_trips.shape)
stop_times_trips.head()

(45558, 25)


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,start_pickup_drop_off_window,end_pickup_drop_off_window,pickup_booking_rule_id,drop_off_booking_rule_id,stop_headsign,...,shape_id,trip_headsign,trip_short_name,direction_id,block_id,agency_id,route_short_name,route_long_name,route_type,rank
0,1.TA.1-MB4-j24-1.1.R,13:34:00,13:34:00,at:48:134,1,NaN,NaN,NaN,NaN,NaN,...,1-MB4-j24-1.1.R,Lindau-Insel,S 5578,1.0,257.0,10,S4,S4,2,1
1,1.TA.1-MB4-j24-1.1.R,13:35:00,13:35:00,at:48:892,2,NaN,NaN,NaN,NaN,NaN,...,1-MB4-j24-1.1.R,Lindau-Insel,S 5578,1.0,257.0,10,S4,S4,2,1
2,1.TA.1-MB4-j24-1.1.R,13:37:00,13:37:00,at:48:139,3,NaN,NaN,NaN,NaN,NaN,...,1-MB4-j24-1.1.R,Lindau-Insel,S 5578,1.0,257.0,10,S4,S4,2,1
3,1.TA.1-MB4-j24-1.1.R,13:40:00,13:40:00,at:48:326,4,NaN,NaN,NaN,NaN,NaN,...,1-MB4-j24-1.1.R,Lindau-Insel,S 5578,1.0,257.0,10,S4,S4,2,1
4,1.TA.1-MB4-j24-1.1.R,13:43:00,13:44:00,at:48:187,5,NaN,NaN,NaN,NaN,NaN,...,1-MB4-j24-1.1.R,Lindau-Insel,S 5578,1.0,257.0,10,S4,S4,2,1


In [37]:
stop_times_grouped = stop_times_trips.groupby(['stop_id']).agg(rank=("rank", "min"), count=("rank", "count")).reset_index().copy()

# TODO: maybe after merge with obb stations
stop_times_grouped["interval"] = stop_times_grouped["count"].apply(lambda x: 840 / (x/2) if x != 0 else 1680)
stop_times_grouped["category"] = stop_times_grouped.apply(lambda x: lookup_category(x["interval"], x["rank"]), axis=1)

stop_times_grouped.sort_values(by=['count'], ascending=False, inplace=True)

stop_times_grouped

,stop_id,rank,count,interval,category
992,at:49:1349,0,837,2.007168,0
980,at:49:1015,0,618,2.718447,0
1001,at:49:1705,0,529,3.175803,0
1015,at:49:560,0,460,3.652174,0
983,at:49:1091,0,392,4.285714,0
...,...,...,...,...,...
672,at:44:49330,0,3,560.000000,-1
739,at:45:56901,1,2,840.000000,-1
390,at:43:7229,1,1,1680.000000,-1
318,at:43:5147,1,1,1680.000000,-1


In [39]:
# TODO: change "how" to "left" to include all stops, also ones without rank and count (nan)
stops_final = pd.merge(stops_filtered_final.drop(['zone_id', 'location_type', 'level_id', 'platform_code', 'parent_station'], axis=1), stop_times_grouped, on='stop_id', how='inner')
stops_final

,stop_id,stop_name,stop_lat,stop_lon,rank,count,interval,category
0,at:41:3087,Bad Sauerbrunn Bahnhof,47.773870,16.324886,1,29,57.931034,4
1,at:41:3145,Baumgarten-Schattendorf Bahnhof,47.727514,16.518257,0,27,62.222222,4
2,at:41:3186,Breitenbrunn Bahnhof,47.941229,16.746385,0,28,60.000000,3
3,at:41:3294,Deutschkreutz Bahnhof,47.603827,16.620054,0,28,60.000000,3
4,at:41:3298,Donnerskirchen Bahnhof,47.888571,16.660416,0,28,60.000000,3
...,...,...,...,...,...,...,...,...
1023,at:49:910,Wien Aspern Nord,48.233852,16.504630,0,112,15.000000,1
1024,at:49:94,Wien Atzgersdorf,48.147016,16.288657,1,168,10.000000,1
1025,at:49:948,Wien Nußdorf,48.260209,16.367664,1,56,30.000000,3
1026,at:49:959,Wien Oberdöbling,48.244011,16.344101,1,163,10.306748,2
